In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/credit-scoring-project/data/application_train.csv')

bureau = pd.read_csv('/content/drive/MyDrive/credit-scoring-project/data/bureau.csv')
prev_app = pd.read_csv('/content/drive/MyDrive/credit-scoring-project/data/previous_application.csv')
installments = pd.read_csv('/content/drive/MyDrive/credit-scoring-project/data/installments_payments.csv')

In [3]:
installments['days_late'] = (
    installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
)

pay_reg = installments.groupby('SK_ID_CURR').agg(
    avg_days_late=('days_late', 'mean'),
    std_days_late=('days_late', 'std'),
    pct_on_time=('days_late', lambda x: (x <= 0).mean())
).reset_index()

In [4]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    num_credit_types=('CREDIT_TYPE', 'nunique'),
    total_credits=('SK_ID_BUREAU', 'count'),
    active_credits=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum())
).reset_index()

In [5]:
prev_agg = prev_app.groupby('SK_ID_CURR').agg(
    num_prev_applications=('SK_ID_PREV', 'count'),
    num_approved=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    num_refused=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    approval_rate=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').mean())
).reset_index()

In [6]:
df = df.merge(pay_reg, on='SK_ID_CURR', how='left')
df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')
df = df.merge(prev_agg, on='SK_ID_CURR', how='left')

print(f"Final dataset shape: {df.shape}")

print(
    "New behavioral features added: "
    "avg_days_late, std_days_late, pct_on_time, "
    "num_credit_types, approval_rate"
)

Final dataset shape: (307511, 132)
New behavioral features added: avg_days_late, std_days_late, pct_on_time, num_credit_types, approval_rate


In [7]:
TRADITIONAL_FEATURES = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'DAYS_BIRTH',
    'DAYS_EMPLOYED',
    'EXT_SOURCE_1',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3'
]

BEHAVIORAL_FEATURES = [
    'avg_days_late',
    'std_days_late',
    'pct_on_time',
    'num_credit_types',
    'approval_rate',
    'num_prev_applications',
    'active_credits'
]

ALL_FEATURES = TRADITIONAL_FEATURES + BEHAVIORAL_FEATURES

print(TRADITIONAL_FEATURES)
print(BEHAVIORAL_FEATURES)

['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
['avg_days_late', 'std_days_late', 'pct_on_time', 'num_credit_types', 'approval_rate', 'num_prev_applications', 'active_credits']


In [8]:
df.to_csv(
    '/content/drive/MyDrive/credit-scoring-project/data/engineered_features.csv',
    index=False
)

In [9]:
import os

os.path.exists(
    '/content/drive/MyDrive/credit-scoring-project/data/engineered_features.csv'
)

True